<a href="https://colab.research.google.com/github/Mitul-Marimuthu/deep-learning/blob/project1/project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

# load MNIST --> 70000 images of handwritten digits
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(int)

#normalize pixel values to [0, 1]
X = X / 255.0

# train.test split
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train: {y_train.shape}, Test: {y_test.shape}")

In [2]:
# initialize weights
np.random.seed(42)

# small weights used because if weights start larger, activations
# saturate and gradients vanish before training ever begins
def init_params():
  # layer 1: 784 inputs --> 128 hidden neurons
  W1 = np.random.randn(784, 128) * 0.01 # small random weights
  b1 = np.zeros((1, 128))

  # Layer 2: 128 hidden -> 10 outputs -> one per digit class
  W2 = np.random.randn(128, 10) * 0.01
  b2 = np.zeros((1, 10))

  return W1, b1, W2, b2

In [3]:
# Forward pass

# gets the relu (rectified linear unit) value
# enables faster training and better gradient flow in
# deep neural networks, mitigating the vanishing gradient
# problem.
def relu(z):
  return np.maximum(0, z)

# turns raw scores into probabilites
def softmax(z):
  # subtract max for numerical stability (prevents overflow)
  z = z - np.max(z, axis=1, keepdims=True) # column max
  exp_z = np.exp(z)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def forward(X, W1, b1, W2, b2):
  # layer 1
  Z1 = X @ W1 + b1 # (batch, 784) @ (784, 128) = (batch, 128)
  A1 = relu(Z1) # apply relu activation

  # Layer 2
  Z2 = A1 @ W2 + b2 # (batch, 128) @ (128, 10) = (batch, 10)
  A2 = softmax(Z2)

  return Z1, A1, Z2, A2

In [4]:
# Loss function
# measures how wrong predictions were
def cross_entropy_loss(A2, y, batch_size):
  # one hot encode labels
  one_hot = np.zeros_like(A2)
  one_hot[np.arange(batch_size), y] = 1

  # cross entropy: -sum(true * log(predicted))
  log_probs = np.log(A2 + 1e-8) # 1e-8 avoids 0
  # highly penalizes confident wrong predications
  loss = -np.sum(one_hot * log_probs) / batch_size # averages loss
  # across all images

  return loss, one_hot

In [6]:
# sanity check for loss function
# Fake a batch of 3 images, 10 classes
A2 = np.array([
    [0.01, 0.01, 0.01, 0.90, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],  # confident, correct (3)
    [0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10],  # totally uncertain
    [0.90, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],  # confident, WRONG (true=3)
])
y = np.array([3, 3, 3])

loss, _ = cross_entropy_loss(A2, y, batch_size=3)
print(f"Loss: {loss:.4f}")
# Confident correct → low, uncertain → medium, confident wrong → high

Loss: 2.3377


In [7]:
# backpropagation
# answers --> which weights are responsible, and in which direction
# should they move to reduce the loss

# computes gradients layer by layer using the chain rule
# dL/dW2 = dL/dA2 * dA2/dZ2 * dZ2/dW2

def backward(X, Z1, A1, A2, W2, one_hot, batch_size):
  dZ2 = A2 - one_hot # dradient of the loss with respect to
  # preactivation of the output layer
  # every wrong class has a positive gradient

  # gradient of loss with respect to W2
  # EVERY GRADIENT MUST BE THE SAME SHAPE OF THE GRADIENT
  # IT IS OF
  dW2 = A1.T @ dZ2 / batch_size
  db2 = np.sum(dZ2, axis=0, keepdims=True) / batch_size

  # Hidden layer
  dA1 = dZ2 @ W2.T
  dZ1 = dA1 * (Z1 > 0)
  dW1 = X.T @ dZ1 / batch_size
  db1 = np.sum(dZ1, axis=0, keepdims=True) / batch_size

  return dW1, db1, dW2, db2

  # Loss
  # ↓  dZ2 = A2 - one_hot          (error at output)
  # ↓  dW2 = A1.T @ dZ2            (how much did W2 cause this?)
  # ↓  dA1 = dZ2 @ W2.T            (propagate error back through W2)
  # ↓  dZ1 = dA1 * (Z1 > 0)        (block gradient through dead ReLUs)
  # ↓  dW1 = X.T @ dZ1             (how much did W1 cause this?)